In [60]:
import pandas as pd
import json
import requests
import glob
from pybaseball import chadwick_register
import os
from functools import reduce

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [70]:
bbraw_root = "./bbref-data/raw"
subfolders = glob.glob(os.path.join(bbraw_root, "*/"))

for yr, subfolder in enumerate(subfolders, start=15):
    print(f"{yr} commencing")
    
    csv_pattern = os.path.join(subfolder, "*.csv")
    csv_files = glob.glob(csv_pattern)
    
    output_path = os.path.join(subfolder, "merged_output.csv")
    
    if output_path in csv_files:
        csv_files.remove(output_path)
        
    if not csv_files:
        continue
        
    print(f"Processing {len(csv_files)} files in: {subfolder}")
    
    df_list = []
    for file in csv_files:
        df = pd.read_csv(file)
        
        # 1. STRIP ALL HIDDEN SPACES from column names (e.g., " -9999" becomes "-9999")
        df.columns = df.columns.str.strip()
        
        # 2. Rename -9999 to Player-additional if necessary
        if "-9999" in df.columns and "Player-additional" not in df.columns:
            df = df.rename(columns={"-9999": "Player-additional"})
            
        # 3. Safety Check: skip the file if it somehow STILL doesn't have the column
        if "Player-additional" not in df.columns:
            print(f"    -> WARNING: Skipping '{file}' (Column missing even after clean)")
            continue
            
        df_list.append(df)
    
    # Skip merging if no valid files exist in this folder
    if not df_list:
        print(f"  -> No valid files to merge for {yr}.")
        continue
    
    # Merge the cleaned dataframes
    merged_df = reduce(
        lambda left, right: pd.merge(
            left, 
            right.drop(columns=[col for col in right.columns if col in left.columns and col != "Player-additional"]), 
            on="Player-additional", 
            how="inner"
        ), 
        df_list
    )
    
    merged_df = merged_df.drop(columns = 'Rk').sort_values(by=["IP"], ascending=False)
    merged_df["Season"] = f"20{yr}"
    
    filename = f"./bbref-data/combined/bbref{yr}FULL.csv"
    
    # Save the merged result safely
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    merged_df.to_csv(filename, index=False)

15 commencing
Processing 2 files in: ./bbref-data/raw/18/
16 commencing
Processing 2 files in: ./bbref-data/raw/20/
17 commencing
Processing 2 files in: ./bbref-data/raw/23/
18 commencing
Processing 2 files in: ./bbref-data/raw/16/
19 commencing
Processing 2 files in: ./bbref-data/raw/15/
20 commencing
Processing 2 files in: ./bbref-data/raw/17/
21 commencing
Processing 2 files in: ./bbref-data/raw/25/
22 commencing
Processing 2 files in: ./bbref-data/raw/19/
23 commencing
Processing 2 files in: ./bbref-data/raw/22/
24 commencing
Processing 2 files in: ./bbref-data/raw/24/
25 commencing
Processing 2 files in: ./bbref-data/raw/21/


In [ ]:
jeson = glob.glob('fg-data/raw/*.json')

fg_col = []

for yr, i in enumerate(jeson, start=15):
    
    with open(i, "r") as file:
        raw_data = json.load(file)
    
    df = pd.json_normalize(raw_data, record_path= ["data"])
    
    df = df.rename(
        columns={
            "WAR": "fWAR",
            "-WPA": "WPA-",
            "+WPA": "WPA+"
        }
    )
    
    fg_col = list(df.columns)
    
    filename= f"fg-data/fg{yr}.csv"
    
    df.to_csv(filename, index=False)

In [5]:
bb15 = pd.read_csv("bbref-data/bbref15.csv")

In [50]:
jeson = glob.glob('bbref-data/raw/*')

bb_col = []

for yr, i in enumerate(jeson, start=15):
    
    df = pd.read_csv(i)
    
    df = df.rename(
        columns={
            "WAR": "bWAR",
            "Player-additional": "bbrefID"
            }
        )
    
    filename = f"bbref-data/bbref{yr}.csv"
    
    bb_col = list(df.columns)
    
    df.to_csv(filename, index=False)

In [21]:
bb_upper = [bb.upper() for bb in bb_col]

fg_upper = [fg.upper() for fg in fg_col]

bb_null = [cols for cols in bb_upper if cols not in fg_upper]

In [33]:
fg_upper

['AVG',
 'AVG+',
 'AGE',
 'AGER',
 'ATTACKANGLE',
 'ATTACKDIRECTION',
 'AVGBATSPEED',
 'BABIP',
 'BABIP+',
 'BB',
 'BB%',
 'BB%+',
 'BB/9',
 'BB/9+',
 'BCORRECT%',
 'BIP-WINS',
 'BK',
 'BOVERTURNED',
 'BPTV',
 'BREVIEW',
 'BS',
 'BS-WINS',
 'BTV',
 'BU',
 'BUH',
 'BUH%',
 'BALLS',
 'BARREL%',
 'BARRELS',
 'BLASTCONTACT%',
 'BLASTSWING%',
 'C+SWSTR%',
 'CB%',
 'CBV',
 'CCORRECT%',
 'CFRAMING',
 'CG',
 'CH%',
 'CHV',
 'COVERTURNED',
 'CPTV',
 'CREVIEW',
 'CSTR%',
 'CT%',
 'CTV',
 'CENT',
 'CENT%',
 'CENT%+',
 'CLUTCH',
 'COMPETITIVESWINGS',
 'CONTACT%',
 'DGV',
 'DSV',
 'DEPTHINBOX',
 'DISTANCEOFFPLATE',
 'DOLLARS',
 'E-F',
 'EBV',
 'ER',
 'ERA',
 'ERA-',
 'ESV',
 'EV',
 'EV90',
 'EVENTS',
 'F-STRIKE%',
 'FB',
 'FB%',
 'FB%+',
 'FB%1',
 'FBV',
 'FIP',
 'FIP-',
 'FASTSWING%',
 'G',
 'GB',
 'GB%',
 'GB%+',
 'GB/FB',
 'GS',
 'GAMES',
 'H',
 'H/9',
 'H/9+',
 'HBP',
 'HLD',
 'HR',
 'HR/9',
 'HR/9+',
 'HR/FB',
 'HRFB%+',
 'HARD',
 'HARD%',
 'HARD%+',
 'HARDHIT',
 'HARDHIT%',
 'IBB',
 'IFFB',
 

In [20]:
fg_col

['AVG',
 'AVG+',
 'Age',
 'AgeR',
 'AttackAngle',
 'AttackDirection',
 'AvgBatSpeed',
 'BABIP',
 'BABIP+',
 'BB',
 'BB%',
 'BB%+',
 'BB/9',
 'BB/9+',
 'BCorrect%',
 'BIP-Wins',
 'BK',
 'BOverturned',
 'BPTV',
 'BReview',
 'BS',
 'BS-Wins',
 'BTV',
 'BU',
 'BUH',
 'BUH%',
 'Balls',
 'Barrel%',
 'Barrels',
 'BlastContact%',
 'BlastSwing%',
 'C+SwStr%',
 'CB%',
 'CBv',
 'CCorrect%',
 'CFraming',
 'CG',
 'CH%',
 'CHv',
 'COverturned',
 'CPTV',
 'CReview',
 'CStr%',
 'CT%',
 'CTv',
 'Cent',
 'Cent%',
 'Cent%+',
 'Clutch',
 'CompetitiveSwings',
 'Contact%',
 'DGV',
 'DSV',
 'DepthInBox',
 'DistanceOffPlate',
 'Dollars',
 'E-F',
 'EBV',
 'ER',
 'ERA',
 'ERA-',
 'ESV',
 'EV',
 'EV90',
 'Events',
 'F-Strike%',
 'FB',
 'FB%',
 'FB%+',
 'FB%1',
 'FBv',
 'FIP',
 'FIP-',
 'FastSwing%',
 'G',
 'GB',
 'GB%',
 'GB%+',
 'GB/FB',
 'GS',
 'Games',
 'H',
 'H/9',
 'H/9+',
 'HBP',
 'HLD',
 'HR',
 'HR/9',
 'HR/9+',
 'HR/FB',
 'HRFB%+',
 'Hard',
 'Hard%',
 'Hard%+',
 'HardHit',
 'HardHit%',
 'IBB',
 'IFFB',
 

In [51]:
bb15 = pd.read_csv("bbref-data/bbref15.csv")

fg15 = pd.read_csv("fg-data/fg15.csv")

In [26]:
fg15.columns

Index(['Throws', 'xMLBAMID', 'Name', 'Team', 'Season', 'Age', 'AgeR',
       'SeasonMin', 'SeasonMax', 'W',
       ...
       'PlayerNameRoute', 'PlayerName', 'positionDB', 'position', 'TeamName',
       'TeamNameAbb', 'teamid', 'playerTeamId', 'playerid', 'EV90'],
      dtype='str', length=544)

In [35]:
playerid_lookup("19427")

,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last


In [39]:
chadwicks = chadwick_player_lu()

NameError: name 'chadwick_player_lu' is not defined

In [45]:
chadwick_table = chadwick_register()

chadwick_table.to_csv("chadwick_table.csv", index=False)

In [55]:
print(fg15.query('playerid == 19427'))

  Throws  xMLBAMID                                               Name  \
0      R    669456  <a href="statss.aspx?playerid=19427&position=P...   

                                                Team  Season   Age     AgeR  \
0  <a href="leaders.aspx?pos=all&stats=pit&lg=all...    2020  25.0  25 - 25   

   SeasonMin  SeasonMax    W    L      ERA     G    GS    QS   CG  ShO   SV  \
0       2020       2020  8.0  1.0  1.62931  12.0  12.0  10.0  0.0  0.0  0.0   

    BS    IP    TBF     H     R    ER   HR    BB  IBB  HBP   WP   BK     SO  \
0  0.0  77.1  297.0  46.0  15.0  14.0  7.0  21.0  0.0  1.0  5.0  0.0  122.0   

     GB    FB    LD  IFFB  Pitches  Balls  Strikes    RS  IFH   BU  BUH  \
0  74.0  46.0  33.0   4.0   1238.0  453.0    785.0  32.0  7.0  0.0  0.0   

         K/9      BB/9      K/BB       H/9      HR/9       AVG      WHIP  \
0  14.198274  2.443965  5.809524  5.353448  0.814655  0.167273  0.866379   

      BABIP      LOB%       FIP     GB/FB       LD%      GB%       FB%  

In [ ]:
print(bb15.query('bbrefID == "biebesh01"'))

        Rk        Player   Age Team  Lg  bWAR    W    L   W-L%   ERA    G  \
398  399.0  Shane Bieber  30.0  TOR  AL   0.8  4.0  2.0  0.667  3.57  7.0   

      GS   GF   CG  SHO   SV    IP     H     R    ER   HR   BB  IBB    SO  \
398  7.0  0.0  0.0  0.0  0.0  40.1  34.0  16.0  16.0  8.0  7.0  0.0  37.0   

     HBP   BK   WP     BF   ERA+   FIP   WHIP   H9  HR9  BB9  SO9  SO/BB  \
398  1.0  0.0  0.0  159.0  121.0  4.47  1.017  7.6  1.8  1.6  8.3   5.29   

    Awards    bbrefID  
398    NaN  biebesh01  
